# Library RNN baseline notebook

This notebook builds a standard-library recurrent baseline using PyTorch `nn.RNN` to compare the C implementation against a common framework-based approach.

## Baseline Intent

- This notebook is a methodological baseline, not a direct line-by-line mirror of the C code.
- It keeps the same dataset, sequence length, 80/20 split, train-only normalization, and evaluation metrics so the comparison stays as fair as possible.
- Core settings: input size 41, hidden size 64, output size 1, sequence length 8, batch size 16, learning rate 0.001, gradient clip 5.0, and 10 epochs.
- Using SGD to better match the C update rule.
- Added gradient clipping with the same clip value used in C.
- Overrode PyTorch default initialization with uniform `[-0.1, 0.1]` for weights and zero bias.


In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

HIDDEN_SIZE = 64
OUTPUT_SIZE = 1
SEQ_LEN = 8
BATCH_SIZE = 16
LEARNING_RATE = 0.001
GRAD_CLIP = 5.0
EPSILON = 1e-7
NUM_EPOCHS = 10
SEED = 42

DATASET_PATH = r'C:\Users\achiu\Documents\Thesis\CIC_Dataset\combined.csv'
if DATASET_PATH is None:
    raise FileNotFoundError('Could not find file.')

print('Using dataset:', DATASET_PATH)
np.set_printoptions(suppress=True, precision=6)
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using dataset: C:\Users\achiu\Documents\Thesis\CIC_Dataset\combined.csv
Using device: cuda


In [2]:
df = pd.read_csv(DATASET_PATH)
df.columns = df.columns.str.strip()

print(df.shape)
print(df.columns[-10:])
print(df.tail())
print(df.iloc[:, -1].astype(str).str.strip().value_counts().head(30))

(2830743, 80)
Index(['min_seg_size_forward', 'Active Mean', 'Active Std', 'Active Max',
       'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min', 'Label'],
      dtype='object')
         Unnamed: 0  Destination Port  Flow Duration  Total Fwd Packets  \
2830738     2830738             61374             61                  1   
2830739     2830739             61378             72                  1   
2830740     2830740             61375             75                  1   
2830741     2830741             61323             48                  2   
2830742     2830742             61326             68                  1   

         Total Backward Packets  Total Length of Fwd Packets  \
2830738                       1                            6   
2830739                       1                            6   
2830740                       1                            6   
2830741                       0                           12   
2830742                       1        

In [3]:
df = pd.read_csv(DATASET_PATH)
print('Raw:', df.shape, df.columns[:5].tolist() + df.columns[-3:].tolist())

df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df.columns = df.columns.str.strip()

if 'Label' not in df.columns:
    raise ValueError(f"Expected a Label column, got: {df.columns.tolist()[-5:]}")

raw_labels = df['Label'].astype(str).str.strip()

numeric = pd.to_numeric(raw_labels, errors='coerce')

if numeric.notna().all():
    labels = numeric.astype(int).to_numpy()

    unique_vals = set(np.unique(labels))
    if not unique_vals.issubset({0, 1}):
        raise ValueError(f"Numeric labels are not binary 0/1: {unique_vals}")
else:
    labels = (
        raw_labels
        .str.upper()
        .apply(lambda x: 0 if x == "BENIGN" else 1)
        .astype(np.int64)
        .to_numpy()
    )

if len(np.unique(labels)) < 2:
    raise ValueError("Only one class after mapping. You are loading the wrong/broken CSV.")

df = df.drop(columns=['Label'])
df = df.select_dtypes(include=[np.number])
df.replace([np.inf, -np.inf], np.nan, inplace=True)

valid_mask = ~df.isna().any(axis=1)
df = df.loc[valid_mask]
labels = labels[valid_mask.to_numpy()]

df.columns = [str(i) for i in range(len(df.columns))]
INPUT_SIZE = len(df.columns)
features = df[[str(i) for i in range(INPUT_SIZE)]].to_numpy(dtype=np.float32)

print('Processed:', df.shape, 'INPUT_SIZE=', INPUT_SIZE)
print('Label counts:', dict(zip(*np.unique(labels, return_counts=True))))
print('Feature NaN count:', np.isnan(features).sum())
print('Feature Inf count:', np.isinf(features).sum())

Raw: (2830743, 80) ['Unnamed: 0', ' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', ' Idle Max', ' Idle Min', ' Label']
Processed: (2827876, 78) INPUT_SIZE= 78
Label counts: {0: 2271320, 1: 556556}
Feature NaN count: 0
Feature Inf count: 0


In [4]:
def build_sequences(features, labels, seq_len=SEQ_LEN):
    n_rows = len(features)
    n_seq = n_rows - seq_len + 1
    if n_seq <= 0:
        raise ValueError(f'Not enough rows to build sequences of length {seq_len}')
    X = np.empty((n_seq, seq_len, features.shape[1]), dtype=np.float32)
    y = np.empty((n_seq,), dtype=np.int64)
    for start in range(n_seq):
        X[start] = features[start:start + seq_len]
        y[start] = labels[start + seq_len - 1]
    return X, y

X_all, y_all = build_sequences(features, labels)
print('Built sequences:', len(X_all))

Built sequences: 2827869


In [5]:
def build_sequences(features, labels, seq_len=SEQ_LEN):
    n_rows = len(features)
    n_seq = n_rows - seq_len + 1
    if n_seq <= 0:
        raise ValueError(f'Not enough rows to build sequences of length {seq_len}')
    X = np.empty((n_seq, seq_len, features.shape[1]), dtype=np.float32)
    y = np.empty((n_seq,), dtype=np.int64)
    for start in range(n_seq):
        X[start] = features[start:start + seq_len]
        y[start] = labels[start + seq_len - 1]
    return X, y

X_all, y_all = build_sequences(features, labels)
print('Built sequences:', len(X_all))

Built sequences: 2827869


In [6]:
# Match the C data pipeline: single shuffle, then 80/20 split, then train-only normalization

# Shuffle
rng = np.random.RandomState(SEED)
perm = rng.permutation(len(X_all))
X_all = X_all[perm]
y_all = y_all[perm]

train_size = int(0.8 * len(X_all))
test_size = len(X_all) - train_size
X_train, y_train = X_all[:train_size], y_all[:train_size]
X_test, y_test = X_all[train_size:], y_all[train_size:]

mean = X_train.mean(axis=(0, 1), dtype=np.float64).astype(np.float32)
var = X_train.astype(np.float64).var(axis=(0, 1)).astype(np.float32)
var = np.maximum(var, 1e-12)
std = np.sqrt(var).astype(np.float32)

X_train = ((X_train - mean) / (std + EPSILON)).astype(np.float32)
X_test = ((X_test - mean) / (std + EPSILON)).astype(np.float32)

train_neg = int(np.sum(y_train == 0))
train_pos = int(np.sum(y_train == 1))
pos_weight = np.float32(train_neg / train_pos) if train_pos > 0 else np.float32(1.0)

print('Train sequences:', train_size)
print('Test sequences:', test_size)
print('Train class counts -> neg:', train_neg, 'pos:', train_pos, '| pos_weight:', float(pos_weight))

Train sequences: 2262295
Test sequences: 565574
Train class counts -> neg: 1816700 pos: 445595 | pos_weight: 4.077020645141602


In [7]:
train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train.astype(np.float32)))
test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test.astype(np.float32)))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [8]:
class LibraryRNNBaseline(nn.Module):
    def __init__(self, input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            nonlinearity='tanh',
            batch_first=True,
            num_layers=1,
            bias=True,
        )
        self.out = nn.Linear(hidden_size, 1)
        self.reset_parameters_like_c()

    def reset_parameters_like_c(self):
        with torch.no_grad():
            for name, param in self.rnn.named_parameters():
                if 'weight' in name:
                    param.uniform_(-0.1, 0.1)
                elif 'bias' in name:
                    param.zero_()
            self.out.weight.uniform_(-0.1, 0.1)
            self.out.bias.zero_()

    def forward(self, x):
        rnn_out, _ = self.rnn(x)
        last_hidden = rnn_out[:, -1, :]
        logits = self.out(last_hidden).squeeze(-1)
        return logits

model = LibraryRNNBaseline().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(pos_weight), device=device))
criterion_eval = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)
model

LibraryRNNBaseline(
  (rnn): RNN(78, 64, batch_first=True)
  (out): Linear(in_features=64, out_features=1, bias=True)
)

In [9]:
def safe_div(num, den):
    return 0.0 if den == 0 else num / den

def evaluate_model(model, loader):
    model.eval()
    total_loss = 0.0
    all_scores = []
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion_eval(logits, yb)
            total_loss += float(loss.item()) * len(xb)

            scores = torch.sigmoid(logits)
            preds = (scores >= 0.5).to(torch.int64)
            all_scores.append(scores.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_targets.append(yb.cpu().numpy().astype(np.int64))

    scores = np.concatenate(all_scores)
    preds = np.concatenate(all_preds)
    targets = np.concatenate(all_targets)

    tp = int(np.sum((targets == 1) & (preds == 1)))
    tn = int(np.sum((targets == 0) & (preds == 0)))
    fp = int(np.sum((targets == 0) & (preds == 1)))
    fn = int(np.sum((targets == 1) & (preds == 0)))

    acc = 100.0 * safe_div(int(np.sum(preds == targets)), len(targets))
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = safe_div(2.0 * precision * recall, precision + recall)
    avg_loss = safe_div(total_loss, len(targets))

    return {
        'loss': avg_loss,
        'acc': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'tp': tp,
        'scores': scores,
        'preds': preds,
        'targets': targets,
    }

In [10]:
def train_and_evaluate_library_rnn(model, train_loader, test_loader, num_epochs=NUM_EPOCHS):
    history = []
    prediction_rows = []
    wall_start = time.time()

    for epoch in range(num_epochs):
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_count = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_value_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            with torch.no_grad():
                scores = torch.sigmoid(logits)
                preds = (scores >= 0.5).to(torch.int64)
                train_correct += int((preds == yb.to(torch.int64)).sum().item())
                train_loss_sum += float(loss.item()) * len(xb)
                train_count += len(xb)

        train_acc = 100.0 * safe_div(train_correct, train_count)
        avg_train_loss = safe_div(train_loss_sum, train_count)

        test_metrics = evaluate_model(model, test_loader)
        elapsed_wall = time.time() - wall_start

        for i in range(min(10, len(test_metrics['targets']))):
            prediction_rows.append({
                'epoch': epoch + 1,
                'sample': i + 1,
                'true_label': int(test_metrics['targets'][i]),
                'pred_label': int(test_metrics['preds'][i]),
                'pred_score': float(test_metrics['scores'][i]),
            })

        row = {
            'epoch': epoch + 1,
            'train_loss': avg_train_loss,
            'train_acc': train_acc,
            'test_loss': test_metrics['loss'],
            'test_acc': test_metrics['acc'],
            'precision': test_metrics['precision'],
            'recall': test_metrics['recall'],
            'f1': test_metrics['f1'],
            'tn': test_metrics['tn'],
            'fp': test_metrics['fp'],
            'fn': test_metrics['fn'],
            'tp': test_metrics['tp'],
            'elapsed_sec': elapsed_wall,
        }
        history.append(row)

        print(f"Epoch {epoch + 1} TRAIN complete | Samples {train_count} | Avg Loss {avg_train_loss:.6f} | Accuracy {train_acc:.2f}%")
        print(f"Epoch {epoch + 1} TEST complete | Samples {len(test_metrics['targets'])} | Avg Loss {test_metrics['loss']:.6f} | Accuracy {test_metrics['acc']:.2f}% | Precision {test_metrics['precision']:.4f} | Recall {test_metrics['recall']:.4f} | F1 {test_metrics['f1']:.4f} | Elapsed {elapsed_wall:.2f} sec")
        print('Confusion Matrix (TEST):')
        print(' Pred 0 Pred 1')
        print(f"True 0 {test_metrics['tn']:8d} {test_metrics['fp']:8d}")
        print(f"True 1 {test_metrics['fn']:8d} {test_metrics['tp']:8d}\n")

    return pd.DataFrame(history), pd.DataFrame(prediction_rows)
    #redictions_df.head(20)

In [11]:
history_df, predictions_df = train_and_evaluate_library_rnn(model, train_loader, test_loader, num_epochs=NUM_EPOCHS)
history_df

Epoch 1 TRAIN complete | Samples 2262295 | Avg Loss 0.164150 | Accuracy 96.96%
Epoch 1 TEST complete | Samples 565574 | Avg Loss 0.069038 | Accuracy 97.92% | Precision 0.9273 | Recall 0.9697 | F1 0.9481 | Elapsed 446.90 sec
Confusion Matrix (TEST):
 Pred 0 Pred 1
True 0   446181     8432
True 1     3357   107604

Epoch 2 TRAIN complete | Samples 2262295 | Avg Loss 0.105871 | Accuracy 97.95%
Epoch 2 TEST complete | Samples 565574 | Avg Loss 0.059169 | Accuracy 98.09% | Precision 0.9295 | Recall 0.9768 | F1 0.9525 | Elapsed 911.45 sec
Confusion Matrix (TEST):
 Pred 0 Pred 1
True 0   446389     8224
True 1     2579   108382

Epoch 3 TRAIN complete | Samples 2262295 | Avg Loss 0.088399 | Accuracy 98.16%
Epoch 3 TEST complete | Samples 565574 | Avg Loss 0.050692 | Accuracy 98.24% | Precision 0.9330 | Recall 0.9809 | F1 0.9563 | Elapsed 1368.30 sec
Confusion Matrix (TEST):
 Pred 0 Pred 1
True 0   446792     7821
True 1     2119   108842

Epoch 4 TRAIN complete | Samples 2262295 | Avg Loss 0.

,epoch,train_loss,train_acc,test_loss,test_acc,precision,recall,f1,tn,fp,fn,tp,elapsed_sec
0,1,0.164150,96.956011,0.069038,97.915569,0.927333,0.969746,0.948065,446181,8432,3357,107604,446.895077
1,2,0.105871,97.952654,0.059169,98.089905,0.929472,0.976758,0.952528,446389,8224,2579,108382,911.451955
2,3,0.088399,98.156430,0.050692,98.242493,0.932961,0.980903,0.956331,446792,7821,2119,108842,1368.300803
3,4,0.076281,98.390440,0.044557,98.443882,0.940041,0.983409,0.961236,447653,6960,1841,109120,1836.114597
4,5,0.068355,98.564864,0.039277,98.629534,0.947680,0.984499,0.965739,448582,6031,1720,109241,2304.222803
5,6,0.063002,98.673162,0.035509,98.757545,0.953148,0.985094,0.968858,449240,5373,1654,109307,2766.933565
6,7,0.058941,98.750251,0.032830,98.848603,0.956999,0.985599,0.971088,449699,4914,1598,109363,4071.875556
7,8,0.055549,98.819694,0.030884,98.909250,0.959292,0.986256,0.972587,449969,4644,1525,109436,5637.128070
8,9,0.052654,98.880473,0.029408,98.968128,0.961394,0.987040,0.974049,450215,4398,1438,109523,7126.673579
9,10,0.050201,98.935462,0.028217,99.020995,0.963427,0.987590,0.975359,450453,4160,1377,109584,8603.927241
